# Robustness check — rolling temporal splits (AAPL)

Requested by the supervisor: re-run the main pipeline on different temporal splits of the
*same, already-downloaded* dataset, to check whether the central result (Black--Scholes
winning in the money, XGBoost winning out of the money) is specific to the original
split or holds more generally.

Design choices, to keep this comparable to the main analysis and cheap to run:
- **Rolling, non-overlapping windows** (not expanding): each split uses roughly the same
  amount of training data, so a difference across splits can be attributed to *which*
  period the model was trained/tested on, not to *how much* data it saw.
- **Same five inputs, same v4 target** (`log(C/K)`) as the main model.
- **Same hyperparameters** already selected via rolling-window CV on the original split
  (Table 10) — no new grid search, to keep this fast.
- **XGBoost only.** The neural network is not re-run here; if time allows, the same
  structure below can be reused for it.
- Only three non-overlapping splits fit inside the ~5 years of data actually downloaded
  (31 Aug 2020 -- 29 Aug 2025), so training windows here are shorter (~12 months) than in
  the main analysis (~3.5 years). Absolute error levels are therefore not directly
  comparable to the main results table; what is comparable is the **pattern across
  moneyness buckets** within and across these three splits.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

import joblib


In [ ]:
aapl = pd.read_csv("AAPL_cleaned.csv")
aapl["date"] = pd.to_datetime(aapl["date"])
aapl["exdate"] = pd.to_datetime(aapl["exdate"])

print(aapl.shape)
aapl.head()


(838276, 19)


,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price,volatility,rate,days_to_maturity,T,last_dividend,frequency,q
0,101594,2020-08-31,2020-09-04,100.00,29.4,29.7,150,411,135212375,129.04,1.290400,29.55,0.375779,0.002148,4,0.010959,0.205,3,0.004766
1,101594,2020-08-31,2020-11-20,80.00,50.1,50.5,0,343,134244951,129.04,1.613000,50.30,0.375779,0.002148,81,0.221918,0.205,3,0.004766
2,101594,2020-08-31,2020-11-20,81.25,48.9,49.3,1,416,134244952,129.04,1.588185,49.10,0.375779,0.002148,81,0.221918,0.205,3,0.004766
3,101594,2020-08-31,2020-11-20,82.50,47.7,48.1,16,383,134244953,129.04,1.564121,47.90,0.375779,0.002148,81,0.221918,0.205,3,0.004766
4,101594,2020-08-31,2020-11-20,83.75,46.5,46.9,0,414,134244954,129.04,1.540776,46.70,0.375779,0.002148,81,0.221918,0.205,3,0.004766


## Shared settings (identical to the main notebook)

In [ ]:
BINS   = [0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100]
LABELS = ["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]

def bucket_mae(df, preds):
    """MAE by moneyness bucket for BS and each model; last row = all options."""
    t = pd.DataFrame({"bucket": pd.cut(df["moneyness (S/K)"], bins=BINS, labels=LABELS),
                      "BS": np.abs(df["price"].values - df["bs_price"].values)})
    for k, p in preds.items():
        t[k] = np.abs(df["price"].values - np.asarray(p))
    out = t.groupby("bucket", observed=True).mean()
    out.insert(0, "n", t.groupby("bucket", observed=True).size())
    out.loc["ALL"] = [len(t)] + list(t.drop(columns="bucket").mean())
    return out.astype({"n": int})

cv_feature_cols = ["moneyness (S/K)", "T", "rate", "q", "volatility"]
cv_target_col = "price"


## 1. Black-Scholes benchmark

Identical formula and inputs as the main notebook. BS price does not depend on the
split, so it is computed once on the full dataset.


In [ ]:
d1 = (
    np.log(aapl["close"] / aapl["strike_price"])
    + (aapl["rate"] - aapl["q"] + 0.5 * aapl["volatility"] ** 2) * aapl["T"]
) / (aapl["volatility"] * np.sqrt(aapl["T"]))

d2 = d1 - aapl["volatility"] * np.sqrt(aapl["T"])

aapl["bs_price"] = (
    aapl["close"] * np.exp(-aapl["q"] * aapl["T"]) * norm.cdf(d1)
    - aapl["strike_price"] * np.exp(-aapl["rate"] * aapl["T"]) * norm.cdf(d2)
)

aapl["bs_price"].describe()


,bs_price
count,838276.000000
mean,34.851495
std,42.635250
min,0.000000
25%,1.404682
50%,16.424608
75%,57.070649
max,253.989479


## 2. Reused hyperparameters (from Table 10, main analysis)

No new grid search: the goal is to test robustness of the *result*, not to re-tune the
model for each split.


In [ ]:
best_params = {"max_depth": 5, "learning_rate": 0.01, "n_estimators": 500, "subsample": 1.0}
best_params = {k: (int(v) if k in ["max_depth", "n_estimators"] else float(v)) for k, v in best_params.items()}
print(best_params)


{'max_depth': 5, 'learning_rate': 0.01, 'n_estimators': 500, 'subsample': 1.0}


## 3. Three rolling, non-overlapping splits

Same length pattern (train / validation / test) repeated three times across the sample,
sliding forward. Split A is close to, but not identical to, the split used in the main
analysis (that one used quantile cutoffs at 70%/85% of the *full* 5-year sample; here
each split is confined to its own ~20-month block so three non-overlapping splits fit).


In [ ]:
splits = {
    "C": {
        "train": ("2020-08-31", "2021-08-31"),
        "val":   ("2021-09-01", "2022-01-01"),
        "test":  ("2022-01-02", "2022-04-30"),
    },
    "B": {
        "train": ("2022-05-01", "2023-05-01"),
        "val":   ("2023-05-02", "2023-09-01"),
        "test":  ("2023-09-02", "2023-12-31"),
    },
    "A": {
        "train": ("2024-01-01", "2025-01-01"),
        "val":   ("2025-01-02", "2025-05-01"),
        "test":  ("2025-05-02", "2025-08-29"),
    },
}

for name, s in splits.items():
    print(name, s)


C {'train': ('2020-08-31', '2021-08-31'), 'val': ('2021-09-01', '2022-01-01'), 'test': ('2022-01-02', '2022-04-30')}
B {'train': ('2022-05-01', '2023-05-01'), 'val': ('2023-05-02', '2023-09-01'), 'test': ('2023-09-02', '2023-12-31')}
A {'train': ('2024-01-01', '2025-01-01'), 'val': ('2025-01-02', '2025-05-01'), 'test': ('2025-05-02', '2025-08-29')}


In [ ]:
te_B = results["B"]["test"]
print("test B rows:", len(te_B))
print(te_B["moneyness (S/K)"].describe())
print()
print("rows with moneyness > 5:", (te_B["moneyness (S/K)"] > 5).sum())
print()
print(pd.cut(te_B["moneyness (S/K)"], bins=BINS, labels=LABELS).value_counts())

test B rows: 42995
count    42995.000000
mean         1.286807
std          0.639327
min          0.521531
25%          0.862667
50%          1.076121
75%          1.486000
max          3.962200
Name: moneyness (S/K), dtype: float64

rows with moneyness > 5: 0

moneyness (S/K)
Deep ITM         11296
Deep OTM          7793
OTM               7643
ITM               5876
Very Deep ITM     5524
ATM               4863
Extreme ITM          0
Name: count, dtype: int64


## 4. Fit and evaluate each split

Same recipe as the main notebook's final model: fit on train+val with `log(C/K)` as
target, evaluate on test, convert predictions back with `exp(pred) * K`.


In [ ]:
def fit_eval_split(df, split_dates, label):
    tr = df[(df["date"] >= split_dates["train"][0]) & (df["date"] <= split_dates["train"][1])]
    va = df[(df["date"] >= split_dates["val"][0])   & (df["date"] <= split_dates["val"][1])]
    te = df[(df["date"] >= split_dates["test"][0])  & (df["date"] <= split_dates["test"][1])]
    tv = pd.concat([tr, va]).sort_values("date").reset_index(drop=True)
    te = te.sort_values("date").reset_index(drop=True)

    print(f"[{label}] train {len(tr):,} | val {len(va):,} | train+val {len(tv):,} | test {len(te):,}")
    print(f"[{label}] test window: {te['date'].min()} to {te['date'].max()}")

    X_tv = tv[cv_feature_cols]
    y_tv_log = np.log(tv[cv_target_col] / tv["strike_price"])

    model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    model.fit(X_tv, y_tv_log)

    X_te = te[cv_feature_cols]
    test_pred = np.exp(model.predict(X_te)) * te["strike_price"].values

    return te, test_pred, model

results = {}
for name, dates in splits.items():
    te, pred, model = fit_eval_split(aapl, dates, f"AAPL-{name}")
    results[name] = {"test": te, "pred": pred, "model": model}


[AAPL-C] train 191,473 | val 54,919 | train+val 246,392 | test 51,422
[AAPL-C] test window: 2022-01-03 00:00:00 to 2022-04-29 00:00:00
[AAPL-B] train 159,654 | val 48,826 | train+val 208,480 | test 42,995
[AAPL-B] test window: 2023-09-05 00:00:00 to 2023-12-29 00:00:00
[AAPL-A] train 165,967 | val 63,434 | train+val 229,401 | test 59,586
[AAPL-A] test window: 2025-05-02 00:00:00 to 2025-08-29 00:00:00


## 5. Per-bucket MAE, each split

In [ ]:
bucket_tables = {}
for name, r in results.items():
    bucket_tables[name] = bucket_mae(r["test"], {"XGB": r["pred"]})
    print(f"--- Split {name} ---")
    print(bucket_tables[name])
    print()


--- Split C ---
                   n        BS       XGB
bucket                                  
Deep OTM        9550  0.216962  1.351206
OTM             8267  0.483364  2.165958
ATM             5161  0.855056  1.885166
ITM             6353  1.006506  1.111552
Deep ITM       14103  1.015744  0.795235
Very Deep ITM   7459  0.398344  2.234156
Extreme ITM      529  0.284916  5.895892
ALL            51422  0.667462  1.528523

--- Split B ---
                   n        BS       XGB
bucket                                  
Deep OTM        7793  0.120088  0.103049
OTM             7643  0.418076  0.383740
ATM             4863  0.949745  0.848143
ITM             5876  1.046583  0.646787
Deep ITM       11296  0.545793  1.335346
Very Deep ITM   5524  0.177131  4.163464
ALL            42995  0.512694  1.156973

--- Split A ---
                   n        BS        XGB
bucket                                   
Deep OTM       13732  2.142575   0.170651
OTM            10398  2.929515   0.696768
ATM

## 6. Side-by-side comparison

The check that matters: does XGBoost beat BS on out-of-the-money/at-the-money buckets,
and does BS beat XGBoost on in-the-money buckets, in *all three* splits — or only in
the original one?


In [ ]:
summary = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    summary[f"BS_{name}"] = t["BS"]
    summary[f"XGB_{name}"] = t["XGB"]

summary.round(3)


,BS_C,XGB_C,BS_B,XGB_B,BS_A,XGB_A
Deep OTM,0.217,1.351,0.120,0.103,2.143,0.171
OTM,0.483,2.166,0.418,0.384,2.930,0.697
ATM,0.855,1.885,0.950,0.848,3.422,1.275
ITM,1.007,1.112,1.047,0.647,2.806,0.933
Deep ITM,1.016,0.795,0.546,1.335,1.235,1.143
Very Deep ITM,0.398,2.234,0.177,4.163,0.282,3.249
Extreme ITM,0.285,5.896,NaN,NaN,0.262,18.347
ALL,0.667,1.529,0.513,1.157,2.028,1.706


In [ ]:
# Same comparison expressed as a ratio (XGB error / BS error): <1 means XGB wins
ratio = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    ratio[name] = t["XGB"] / t["BS"]

ratio.round(3)


,C,B,A
Deep OTM,6.228,0.858,0.080
OTM,4.481,0.918,0.238
ATM,2.205,0.893,0.373
ITM,1.104,0.618,0.332
Deep ITM,0.783,2.447,0.925
Very Deep ITM,5.609,23.505,11.511
Extreme ITM,20.693,NaN,70.069
ALL,2.290,2.257,0.841


## 7. Save results


In [ ]:
joblib.dump(
    {"splits": splits, "best_params": best_params, "summary": summary, "ratio": ratio,
     "bucket_tables": bucket_tables},
    "robustness_aapl_results.pkl"
)


['robustness_aapl_results.pkl']